# 09 · 例外与文件 I/O

本书教你在程序出错时优雅地处理例外，并用正确的方式读写各种格式的文件与管理路径。

## 学习目标
- 理解例外处理（exception handling）的流程，能用 try/except 拦截并回应错误而不让程序中断。
- 能使用 with 上下文管理器（context manager）安全地管理文件等资源，确保正确释放。
- 掌握纯文字档的读写，并理解编码（encoding）对中文等文字的重要性。
- 能用对象式路径（pathlib.Path）与传统 os/os.path 操作文件系统，并用 glob 搜索文件。
- 能在 JSON 与 CSV 这两种常见数据格式之间进行串行化（serialization）与还原。
- 认识 warnings 警告控制与其他串行化工具（yaml、pickle、base64）的适用情境。

## 例外处理基础

例外（exception）是程序运行中发生的错误事件，例如除以零或类型不符。预设情况下未处理的例外会让程序整个中断。

程序出错是常态，学会拦截特定例外并给予合理回应，比让程序整个崩溃更稳健。

形状：
```
try:
    可能出错的程序
except 某种例外类别 as e:
    出错时的处理
else:
    没出错才运行
finally:
    无论如何都运行
```

常见内置例外对照：

| 例外类别 | 触发时机 |
|---|---|
| ZeroDivisionError | 除以零 |
| TypeError | 类型不符的运算 |
| ValueError | 类型对但值不合法 |
| Exception | 几乎所有例外的基底类别 |

In [ ]:
# 概念：用 try/except 拦截特定例外，回传友善消息而非让程序崩溃
def safe_divide(a, b):
    try:
        result = a / b                      # 可能引发 ZeroDivisionError 或 TypeError
    except ZeroDivisionError:               # 多重 except：先拦最具体的例外
        return "错误：除数不能为零"
    except TypeError:                       # 输入不是数字时触发
        return "错误：输入必须是数字"
    else:
        return f"结果是 {result}"          # 没出错才运行 else
    finally:
        # 注意：finally 无论成功或失败都会运行，常用来收尾
        print("  （已完成一次除法尝试）")

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(10, "x"))

# 概念：用 raise 主动抛出例外，表达「这个输入我不接受」
def check_height(h):
    if h <= 0:
        raise ValueError("楼高必须为正数")   # 主动抛出，交给调用端处理
    return h

try:
    check_height(-5)
except ValueError as e:                      # as e 取得例外对象，可读消息
    print("拦截到：", e)

## 上下文管理器与 warnings

上下文管理器（context manager）是搭配 with 使用的对象，能在进入区块时取得资源、离开时自动释放，即使中途发生例外也会释放。

with 能保证资源（如文件）被正确关闭；warnings 用于提示而非中断，与例外做区隔。

形状：
```
with 上下文管理器 as 变量:
    使用资源
# 离开区块自动清理
```

In [ ]:
# 概念：用 contextlib 自制计时上下文管理器，示范进入与离开区块的行为
import time
from contextlib import contextmanager

@contextmanager                              # 把产生器函数变成上下文管理器
def timer(label):
    start = time.perf_counter()              # 进入区块：取得起始时间
    print(f"[{label}] 开始")
    try:
        yield                                # yield 之前是进入、之后是离开
    finally:
        # 技巧：放在 finally 确保区块内出错也会印出耗时
        elapsed = time.perf_counter() - start
        print(f"[{label}] 结束，耗时 {elapsed:.4f} 秒")

with timer("累加循环"):
    total = sum(i for i in range(100000))
    print("  总和 =", total)

# 概念：用 warnings 对弃用功能发出提示，不中断程序
import warnings

def old_area(side):
    # 注意：warnings.warn 只是提示，程序会继续往下跑
    warnings.warn("old_area 已弃用，请改用 square_area", DeprecationWarning)
    return side * side

with warnings.catch_warnings():
    warnings.simplefilter("always")          # 让 DeprecationWarning 每次都显示
    print("面积 =", old_area(4))

## 文字档读写与编码

用 open() 打开文件，透过模式（mode）决定读或写，再用 read/write 操作内容。

读写档最常见的坑就是编码（encoding），明确指定 utf-8 可避免中文乱码；搭配 with 写出安全的读写样板。

常用模式对照：

| 模式 | 意义 |
|---|---|
| r | 读取（文件须存在） |
| w | 写入（覆盖既有内容） |
| a | 附加（接在结尾） |

In [ ]:
# 概念：用 with + open() 指定 utf-8 编码写入中文，再读回
# 造一段内置的多行中文字符串当示范用
text = "第一行：台北101\n第二行：高雄港\n第三行：日月潭\n"

filename = "demo_text.txt"
with open(filename, "w", encoding="utf-8") as f:   # 注意：明确指定 utf-8 才能正确写中文
    f.write(text)                                   # w 模式会覆盖既有内容

# 概念：逐行迭代读档，文件对象本身就可当迭代器
with open(filename, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):           # 一次拿一行，省内存
        print(f"读到第 {i} 行：{line.rstrip()}")    # rstrip 去掉行尾换行符

# 技巧：一次读全部用 read()，读成 list（每元素一行）用 readlines()
with open(filename, "r", encoding="utf-8") as f:
    print("总字符数 =", len(f.read()))

## 路径管理：pathlib 与 os

对象式路径（pathlib.Path）把路径当成对象，用操作符 `/` 组合子路径，并提供 exists、mkdir 等方法。

跨平台时手动拼接路径容易出错，pathlib 提供更直觉安全的写法，并可对照传统 os.path 作法。

形状：
```
Path("文件夹") / "子文件夹" / "文件名.txt"
```

In [ ]:
# 概念：用 pathlib.Path 创建暂存文件夹并组出子文件路径
from pathlib import Path

base = Path("demo_workspace")                # 把路径当对象
base.mkdir(exist_ok=True)                     # 注意：exist_ok=True 避免文件夹已存在时报错

data_file = base / "buildings.txt"           # 用 / 操作符组路径，跨平台安全
data_file.write_text("示范内容\n", encoding="utf-8")   # Path 直接写档，省去 open

print("文件夹存在吗？", base.exists())
print("文件绝对路径：", data_file.resolve())
print("扩展名：", data_file.suffix, "｜文件名：", data_file.name)

# 概念：对照传统 os / os.path 的写法
import os

old_path = os.path.join("demo_workspace", "buildings.txt")   # os.path.join 手动拼接
print("os.path 组出的路径：", old_path)
print("os.path.exists：", os.path.exists(old_path))

## 文件搜索 glob

glob 用通配符（wildcard）样式比对路径，一次取得符合条件的文件清单。

实务上常需要一次处理一批文件，glob 让你用样式比对取得符合条件的路径。

通配符对照：

| 样式 | 意义 |
|---|---|
| `*` | 比对同一层任意字符 |
| `**` | 递归比对所有子层 |

In [ ]:
# 概念：先造出几个不同扩展名的文件，再用通配符样式筛出特定类型
from pathlib import Path

folder = Path("demo_glob")
folder.mkdir(exist_ok=True)

# 造一批仿真文件（两个 csv、两个 txt）当示范用
for name in ["a.csv", "b.csv", "note.txt", "readme.txt"]:
    (folder / name).write_text("x", encoding="utf-8")

# glob：只比对同一层；用 * 当通配符
csv_files = list(folder.glob("*.csv"))       # 筛出所有 .csv
print("找到的 CSV：", [p.name for p in csv_files])

# 技巧：rglob 会递归搜索所有子文件夹，等同 glob('**/...')
all_txt = list(folder.rglob("*.txt"))
print("递归找到的 TXT：", [p.name for p in all_txt])

## JSON 串行化

串行化（serialization）是把内存中的对象转成可保存或传输的文字 / 字节；JSON 是其中最通用的文字格式。

JSON 是设置档与数据交换的通用格式，学会在字典清单与 JSON 文字之间双向转换。

Python 与 JSON 类型对应：

| Python | JSON |
|---|---|
| dict | object |
| list | array |
| str | string |
| True/False/None | true/false/null |

In [ ]:
# 概念：把嵌套字典写成 JSON 档再读回，比较是否一致
import json

# 造一笔内置的嵌套数据当示范用
city = {
    "name": "范例市",
    "buildings": [
        {"name": "市政大楼", "floors": 12},
        {"name": "图书馆", "floors": 5},
    ],
}

with open("city.json", "w", encoding="utf-8") as f:
    # 注意：ensure_ascii=False 中文才不会被转成 \uXXXX；indent 让文件易读
    json.dump(city, f, ensure_ascii=False, indent=2)

with open("city.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)                    # 从文件读回成 Python 对象

print("还原后与原对象相同？", loaded == city)
# 技巧：dumps（多一个 s）回传字符串而非写档，方便调试时直接看内容
print(json.dumps(loaded, ensure_ascii=False))

## CSV 表格数据读写

CSV（逗号分隔值）是以列为单位、字段用分隔符号（delimiter）隔开的纯文字表格格式。

CSV 是试算表与表格数据的常见格式，了解用列为单位与用字段名称两种读写方式。

两种读写器对照：

| 工具 | 每列形式 |
|---|---|
| csv.reader / writer | 用 list（依位置） |
| csv.DictReader / DictWriter | 用 dict（依栏名） |

In [ ]:
# 概念：用 DictWriter 以字段名称写出 CSV，再用 DictReader 读回
import csv

# 造几笔内置人员数据当示范用
people = [
    {"name": "小明", "age": 30, "city": "台北"},
    {"name": "小华", "age": 25, "city": "台中"},
    {"name": "小美", "age": 28, "city": "高雄"},
]
fields = ["name", "age", "city"]             # 表头 header，决定字段顺序

# 注意：newline='' 是 csv 模块的标准写法，避免多出空白列
with open("people.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()                     # 先写表头
    writer.writerows(people)                 # 一次写多列

with open("people.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)               # 每列读成 dict，键就是栏名
    for row in reader:
        # 注意：CSV 读回的值都是字符串，数值要自己 int/float 转换
        print(f"{row['name']} 在 {row['city']}，明年 {int(row['age']) + 1} 岁")

## 其他串行化格式概览

除了 JSON 与 CSV，还有几种常见的串行化工具，各有适用情境与风险。创建选型概念，知道何时该用哪一种。

格式选型对照：

| 格式 | 适用情境 | 风险 / 限制 |
|---|---|---|
| YAML | 人类可读的设置档 | 需第三方套件、缩进敏感 |
| pickle | Python 对象二进位串行化 | 仅限信任来源，可运行任意代码 |
| base64 | 把二进位塞进文字信道 | 不是加密，只是编码 |

In [ ]:
# 概念：把同一份数据概念性地对应到 pickle 与 base64，着重何时该用哪一种
import pickle
import base64

data = {"floors": 10, "tags": ["住宅", "商办"]}   # 造一份内置数据当示范用

# pickle：串行化成 bytes，能保留 Python 对象结构
# 注意：pickle 反串行化会运行内容，只可加载信任来源的数据
blob = pickle.dumps(data)
print("pickle 还原：", pickle.loads(blob))

# base64：把二进位 bytes 编成纯文字，方便塞进 JSON 或网址
# 注意：base64 只是编码不是加密，不提供任何保密性
encoded = base64.b64encode(blob).decode("ascii")
print("base64 文字（前 40 字）：", encoded[:40], "...")

# 技巧：YAML 需 pip install pyyaml；此处示意其风格，设置档常用它因为最易读
yaml_like = "floors: 10\ntags:\n  - 住宅\n  - 商办"
print("YAML 风格示意：\n" + yaml_like)

## 练习

以下三题由浅到深，皆为复合型但简短可快速完成。数据一律自己用 list / numpy 造，不读任何外部文件。只需在 TODO 处完成。

In [ ]:
# TODO 1 ·（简单）建筑清单存成 JSON（集成：文字档读写与编码 + JSON 串行化）
#   情境：把一批建筑的楼高与楼层数仿真数据存成设置档，方便之后读回。
#   要求：
#     1. 在程序内用 list 自己造出 5 笔建筑字典，每笔含中文名称、楼高 height（公尺）、楼层数 floors（整数）。
#     2. 用 with 搭配 open() 并指定 utf-8 编码，以 json.dump 写入文件，设置 ensure_ascii=False 让中文正常显示。
#     3. 再用 with 与 json.load 把文件读回成 Python 对象。
#   验收：读回后的清单与原始 5 笔数据完全一致，且文件内的中文不是乱码。
# TODO: 在这里完成


In [ ]:
# TODO 2 ·（中间）多份街廓 CSV 汇整（集成：glob + CSV 读写 + pathlib + 例外处理）
#   情境：每个街廓 block 一份 CSV，记录该街廓各建筑的楼地板面积 GFA，要汇整成每街廓总量。
#   要求：
#     1. 用 pathlib.Path 创建暂存文件夹，并用内置仿真数据以 csv.DictWriter 写出 3 份 CSV（表头：建筑 id、GFA）。
#     2. 用 pathlib 的 glob 以通配符样式 *.csv 取得所有文件清单。
#     3. 逐档用 csv.DictReader 读入，累加每份文件的 GFA 总和；对读取或数值转换失败者用 try/except 拦截并略过。
#     4. 把每街廓的汇整结果整理成字典。
#   验收：每个街廓文件名对应一个 GFA 总和；故意放入一笔坏数据时程序不中断而是被略过。
# TODO: 在这里完成


In [ ]:
# TODO 3 ·（稍难）容积率政策情境前后比较（集成：JSON + 例外处理 + with 上下文管理器 + 条件聚合）
#   情境：建筑数据含占地面积 footprint 与楼地板面积 GFA，计算容积率 FAR = GFA / footprint，
#         比较套用高度乘数政策前后，有多少栋超出容积上限。
#   要求：
#     1. 用 numpy 造出 8 栋建筑的 footprint 与 floors 仿真数组，并依楼层推算 GFA。
#     2. 写一个函数计算每栋 FAR，对 footprint 为 0 用 try/except 拦截除零并标记为无效，不让程序崩溃。
#     3. 设置容积上限门槛，先统计政策前超标栋数；再对所有 GFA 乘上 1.2 高度乘数仿真放宽，重新统计超标栋数。
#     4. 以 with 开档并用 json.dump（ensure_ascii=False）把「政策前超标数、政策后超标数、各栋 FAR」写成 JSON 报告。
#   验收：政策后超标栋数不少于政策前；JSON 报告能被 json.load 正确读回、含无效栋位的标记。
# TODO: 在这里完成


## 小结

- 例外处理用 try/except/else/finally 拦截特定错误，并可用 raise 主动抛出，让程序在出错时仍可控。
- with 上下文管理器保证资源（尤其是文件）即使中途出错也会释放；warnings 用于提示而非中断。
- 文字档读写务必明确指定 encoding="utf-8"，是避免中文乱码最关键的一步。
- pathlib.Path 以对象式操作符组路径，比手动拼接更安全跨平台，并可对照传统 os.path。
- glob 用通配符一次取得整批文件；JSON 与 CSV 分别是数据交换与表格数据的通用串行化格式。
- 其他格式各有定位：YAML 重可读、pickle 限信任来源、base64 用于把二进位塞进文字信道。